# Calibration Performance Indices

**Purpose:** Calculates standard hydrological model performance metrics
(NSE, RMSE, KGE and its components) for a single calibration run and
displays the results alongside an observed vs. simulated discharge plot.

**What it does:**
- Implements NSE, RMSE, and KGE (with correlation r, bias β, variability γ)
- Reads observed and simulated discharge from an Excel file
- Plots the hydrograph and overlays a metrics summary box

**Input:** `Calibration.xlsx` (observed + simulated discharge)  
**Output:** Hydrograph plot with metrics box

---

In [ ]:
"""
Hydrological Model Performance Metrics
--------------------------------------

This script defines functions for three standard model performance indices:
- Nash–Sutcliffe Efficiency (NSE)
--------------------------------
    Measures how well the simulated discharge reproduces the observed discharge.
    Range: -∞ to 1
        NSE = 1 → perfect fit
        NSE = 0 → model is as good as using the mean of observed data
        NSE < 0 → model is worse than the mean
Nash, J. E., & Sutcliffe, J. V. (1970). River flow forecasting through conceptual models part I — A discussion of principles. Journal of Hydrology, 10(3), 282–290.

- Root Mean Square Error (RMSE)
--------------------------------
    Quantifies the average magnitude of error between observed and simulated values.
    Range: 0 (perfect) to ∞
Althoff, D., & Rodrigues, L. N. (2021). Goodness-of-fit criteria for hydrological models: Model calibration and performance assessment. Journal of Hydrology, 600, 126674.

- Kling–Gupta Efficiency (KGE; revised 2012 version using γ = CV ratio)
-----------------------------------------------------------------------
    Combines correlation (r), bias (β), and variability (γ = CV ratio) components
    for a comprehensive performance assessment.

    Formulation (Kling et al., 2012):
        KGE' = 1 - sqrt( (r - 1)^2 + (γ - 1)^2 + (β - 1)^2 )

        where:
            r  = Pearson correlation coefficient
            γ  = (CV_sim / CV_obs) = (σ_sim/μ_sim) / (σ_obs/μ_obs)
            β  = (μ_sim / μ_obs)

    Range: -∞ to 1 (1 = perfect)


Gupta, H. V., Kling, H., Yilmaz, K. K., & Martinez, G. F. (2009). Decomposition of the mean squared error and NSE performance criteria: Implications for improving hydrological modeling. Water Resources Research, 45(10), W10414.
Kling, H., Fuchs, M., & Paulin, M. (2012). Runoff conditions in the upper Danube basin under an ensemble of climate change scenarios. Hydrology and Earth System Sciences, 16(7), 2529–2545.
"""

In [ ]:
pip install openpyxl

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import matplotlib.dates as mdates
from hydroeval import nse, rmse, kge  

In [ ]:

# 1. Nash–Sutcliffe Efficiency (NSE)
# ----------------------------------------------------------------------
def nse(q_obs, q_sim):
    mask = np.isfinite(q_obs) & np.isfinite(q_sim)
    q_obs, q_sim = q_obs[mask], q_sim[mask]

    if len(q_obs) == 0 or np.all(q_obs == q_obs[0]):
        return np.nan

    numerator = np.sum((q_obs - q_sim) ** 2)
    denominator = np.sum((q_obs - np.mean(q_obs)) ** 2)
    return 1 - (numerator / denominator)

# 2. Root Mean Square Error (RMSE)
# ----------------------------------------------------------------------
def rmse(q_obs, q_sim):
    mask = np.isfinite(q_obs) & np.isfinite(q_sim)
    q_obs, q_sim = q_obs[mask], q_sim[mask]

    if len(q_obs) == 0:
        return np.nan

    return np.sqrt(np.mean((q_obs - q_sim) ** 2))

# 3. Kling–Gupta Efficiency (KGE, revised 2012)
# ----------------------------------------------------------------------
def kge(q_obs, q_sim):
    mask = np.isfinite(q_obs) & np.isfinite(q_sim)
    q_obs, q_sim = q_obs[mask], q_sim[mask]

    if len(q_obs) == 0:
        return np.nan, np.nan, np.nan, np.nan

    # Correlation coefficient
    if np.std(q_obs) == 0 or np.std(q_sim) == 0:
        r = np.nan
    else:
        r = np.corrcoef(q_obs, q_sim)[0, 1]

    mean_obs, mean_sim = np.mean(q_obs), np.mean(q_sim)
    std_obs, std_sim = np.std(q_obs), np.std(q_sim)

    cv_obs = std_obs / mean_obs if mean_obs != 0 else np.nan
    cv_sim = std_sim / mean_sim if mean_sim != 0 else np.nan

    beta = mean_sim / mean_obs if mean_obs != 0 else np.nan
    gamma = cv_sim / cv_obs if cv_obs != 0 else np.nan

    if np.isnan(r) or np.isnan(beta) or np.isnan(gamma):
        return np.nan, r, beta, gamma

    kge_value = 1 - np.sqrt((r - 1) ** 2 + (gamma - 1) ** 2 + (beta - 1) ** 2)
    return kge_value, r, beta, gamma


In [ ]:

def main():
    plt.rcParams.update({
        'font.family':    'monospace',
        'font.size':      9,
        'axes.titlesize': 10,
        'axes.labelsize': 9,
        'xtick.labelsize': 8,
        'ytick.labelsize': 8,
        'legend.fontsize': 8,
    })
    # -------------------------
    merged_file = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Simulation_ZR\Calibration.xlsx"
    df = pd.read_excel(merged_file)

    # -------------------------
    # 2. Convert datetime
    # -------------------------
    df['DateTime'] = pd.to_datetime(df['Zeit'], format="%d.%m.%Y %H:%M", errors='coerce')
    df = df.dropna(subset=['DateTime', 'Obs', 'Sim'])

    # -------------------------
    # 3. Define time window
    # -------------------------
    start_time = pd.to_datetime("2011-11-01 00:00")
    end_time   = pd.to_datetime("2017-10-31 23:00")

    # Filter correctly (your bug was here)
    df = df[(df["DateTime"] >= start_time) & (df["DateTime"] <= end_time)]

    # -------------------------
    # 4. Compute metrics
    # -------------------------
    NSE = nse(df["Obs"].values, df["Sim"].values)
    RMSE = rmse(df["Obs"].values, df["Sim"].values)
    KGE, r, beta, gamma = kge(df["Obs"].values, df["Sim"].values)

    # -------------------------
    # 5. Plot Hydrograph
    # -------------------------
    plt.figure(figsize=(6.5, 4.5))
    plt.plot(df["DateTime"], df["Obs"], label="Observed", color="black", linewidth=1)
    plt.plot(df["DateTime"], df["Sim"], label="Simulated", color="blue", linestyle="--", linewidth=0.7)

    plt.xlabel("Date / Time")
    plt.ylabel("Discharge (m³/s)")
    plt.title("Observed vs Simulated Discharge")

    # -------------------------
    # 6. Add metrics textbox
    # -------------------------
    textstr = (
        f"NSE   = {NSE:.3f}\n"
        f"RMSE  = {RMSE:.3f}\n"
        f"KGE   = {KGE:.3f}\n"
        f"r     = {r:.3f}\n"
        f"β     = {beta:.3f}\n"
        f"γ     = {gamma:.3f}"
    )
    props = dict(boxstyle="round", facecolor="white", alpha=0.8)
    plt.text(0.02, 0.98, textstr, transform=plt.gca().transAxes,
             fontsize=7, verticalalignment='top', bbox=props)

    # -------------------------
    # 7. Format x-axis
    # -------------------------
    plt.xlim(df['DateTime'].min(), df['DateTime'].max())
    plt.gca().xaxis.set_major_locator(mdates.YearLocator())
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.savefig("Simulation.png", dpi=300, bbox_inches='tight')
    plt.show()


if __name__ == "__main__":
    main()

## 

In [ ]:
def main():
    plt.rcParams.update({
        'font.family':    'monospace',
        'font.size':      9,
        'axes.titlesize': 10,
        'axes.labelsize': 9,
        'xtick.labelsize': 8,
        'ytick.labelsize': 8,
        'legend.fontsize': 8,
    })

    # ── Load data ──────────────────────────────────────────────────────────────
    merged_file = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Simulation_ZR\Calibration.xlsx"
    df = pd.read_excel(merged_file)

    # ── Datetime & filter ──────────────────────────────────────────────────────
    df['DateTime'] = pd.to_datetime(df['Zeit'], format="%d.%m.%Y %H:%M", errors='coerce')
    df = df.dropna(subset=['DateTime', 'Obs', 'Sim'])
    df = df[(df["DateTime"] >= pd.to_datetime("2011-11-01 00:00")) &
            (df["DateTime"] <= pd.to_datetime("2017-10-31 23:00"))]

    # ── Metrics ────────────────────────────────────────────────────────────────
    NSE_nr,  RMSE_nr                    = nse(df["Obs"].values, df["Sim"].values),     rmse(df["Obs"].values, df["Sim"].values)
    KGE_nr,  r_nr,  beta_nr,  gamma_nr = kge(df["Obs"].values, df["Sim"].values)

    NSE_wr,  RMSE_wr                    = nse(df["Obs"].values, df["Sim_Res"].values), rmse(df["Obs"].values, df["Sim_Res"].values)
    KGE_wr,  r_wr,  beta_wr,  gamma_wr = kge(df["Obs"].values, df["Sim_Res"].values)

    # ── Zoom window (edit these two lines to change the zoomed event) ──────────
    ZOOM_START = pd.to_datetime("2015-09-01")
    ZOOM_END   = pd.to_datetime("2015-09-30")

    # ── Main figure ────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(6.5, 4.5))

    ax.plot(df["DateTime"], df["Obs"],
            label="Observed", color="black", linewidth=1.2, zorder=1, alpha=0.9)
    ax.plot(df["DateTime"], df["Sim"],
            label="Sim (without reservoir)", color="#2196F3",
            linestyle="--", linewidth=0.9, zorder=2, alpha=0.85)
    ax.plot(df["DateTime"], df["Sim_Res"],
            label="Sim (with reservoir)", color="#d62728",
            linestyle="-", linewidth=1.0, zorder=3, alpha=0.9)

    # ── Main axis ticks: yearly major + monthly minor ──────────────────────────
    ax.set_xlim(df['DateTime'].min(), df['DateTime'].max())
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_minor_locator(mdates.MonthLocator())
    ax.tick_params(axis='x', which='minor', length=3, color='gray')
    ax.tick_params(axis='x', which='major', length=6)
    plt.xticks(rotation=45)

    # ── Draw zoom box on main axis ─────────────────────────────────────────────
    data_max = max(df["Obs"].max(), df["Sim"].max(), df["Sim_Res"].max())
    ax.set_ylim(0, data_max * 1.05)
    y_lo, y_hi = ax.get_ylim()

    rect = plt.Rectangle(
        (mdates.date2num(ZOOM_START), y_lo),
        mdates.date2num(ZOOM_END) - mdates.date2num(ZOOM_START),
        (y_hi - y_lo),
        linewidth=1.2, edgecolor='darkorange', facecolor='orange',
        alpha=0.08, zorder=4
    )
    ax.add_patch(rect)
    for xd in [ZOOM_START, ZOOM_END]:
        ax.axvline(xd, color='darkorange', lw=0.9, ls='--', alpha=0.7, zorder=5)

    # ── Metric boxes ───────────────────────────────────────────────────────────
    text_nr = (
        f"  Without Reservoir  \n"
        f"NSE   = {NSE_nr:.3f}\n"
        f"RMSE  = {RMSE_nr:.3f}\n"
        f"KGE   = {KGE_nr:.3f}\n"
        f"r     = {r_nr:.3f}\n"
        f"β     = {beta_nr:.3f}\n"
        f"γ     = {gamma_nr:.3f}"
    )
    text_wr = (
        f"   With Reservoir    \n"
        f"NSE   = {NSE_wr:.3f}\n"
        f"RMSE  = {RMSE_wr:.3f}\n"
        f"KGE   = {KGE_wr:.3f}\n"
        f"r     = {r_wr:.3f}\n"
        f"β     = {beta_wr:.3f}\n"
        f"γ     = {gamma_wr:.3f}"
    )
    ax.text(0.77, 0.98, text_nr, transform=ax.transAxes, fontsize=7,
            va='top', ha='left',
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.9, edgecolor="#2196F3"))
    ax.text(0.77, 0.73, text_wr, transform=ax.transAxes, fontsize=7,
            va='top', ha='left',
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.9, edgecolor="#d62728"))

    ax.set_xlabel("Date / Time")
    ax.set_ylabel("Discharge (m³/s)")
    ax.set_title("Observed vs Simulated Discharge")

    # ── Inset axes (zoom panel) ────────────────────────────────────────────────
    axins = ax.inset_axes([0.27, 0.57, 0.38, 0.42])

    # ── Inset y-axis on the RIGHT to avoid collision with main y-axis ──────────
    axins.yaxis.set_label_position("right")
    axins.yaxis.tick_right()

    mask = (df["DateTime"] >= ZOOM_START) & (df["DateTime"] <= ZOOM_END)
    dz   = df[mask]

    axins.plot(dz["DateTime"], dz["Obs"],
               color="black", linewidth=1.0)
    axins.plot(dz["DateTime"], dz["Sim"],
               color="#2196F3", linestyle="--", linewidth=0.8)
    axins.plot(dz["DateTime"], dz["Sim_Res"],
               color="#d62728", linestyle="-", linewidth=0.9)

    axins.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))   # every Monday
    axins.xaxis.set_major_formatter(mdates.DateFormatter('%d\n%b'))
    axins.xaxis.set_minor_locator(mdates.DayLocator())
    axins.set_xlim(ZOOM_START, ZOOM_END)
    axins.set_ylabel("m³/s", fontsize=7)
    axins.grid(True, which='major', linestyle='', alpha=0.4)
    axins.patch.set_alpha(0.95)

    # ── Connector lines from zoom box to inset ─────────────────────────────────
    from mpl_toolkits.axes_grid1.inset_locator import mark_inset
    mark_inset(ax, axins, loc1=2, loc2=3, fc="none", ec="darkorange",
               lw=0.8, linestyle='--', alpha=0.7)

    # ── Footer legend (outside plot, centred below x-axis) ────────────────────
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(
        handles, labels,
        loc='lower center',
        bbox_to_anchor=(0.5, -0.02),
        ncol=3,
        fontsize=8,
        frameon=True,
        edgecolor='lightgray',
        fancybox=False,
        handlelength=2.0,
        columnspacing=2.0,
    )

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.18)   # ← increase to 0.22 if legend is clipped
    plt.savefig("Simulation.png", dpi=300, bbox_inches='tight')
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from mpl_toolkits.axes_grid1.inset_locator import mark_inset


# ── Metrics ────────────────────────────────────────────────────────────────────

def nse(obs, sim):
    obs, sim = np.asarray(obs, dtype=float), np.asarray(sim, dtype=float)
    mask = ~(np.isnan(obs) | np.isnan(sim))
    obs, sim = obs[mask], sim[mask]
    return 1 - np.sum((obs - sim) ** 2) / np.sum((obs - np.mean(obs)) ** 2)


def rmse(obs, sim):
    obs, sim = np.asarray(obs, dtype=float), np.asarray(sim, dtype=float)
    mask = ~(np.isnan(obs) | np.isnan(sim))
    obs, sim = obs[mask], sim[mask]
    return np.sqrt(np.mean((obs - sim) ** 2))


def kge(obs, sim):
    obs, sim = np.asarray(obs, dtype=float), np.asarray(sim, dtype=float)
    mask = ~(np.isnan(obs) | np.isnan(sim))
    obs, sim = obs[mask], sim[mask]
    r     = np.corrcoef(obs, sim)[0, 1]
    beta  = np.mean(sim) / np.mean(obs)
    gamma = (np.std(sim) / np.mean(sim)) / (np.std(obs) / np.mean(obs))
    kge_  = 1 - np.sqrt((r - 1)**2 + (beta - 1)**2 + (gamma - 1)**2)
    return kge_, r, beta, gamma


# ── Zoom windows ───────────────────────────────────────────────────────────────
# Edit these four lines to change the highlighted events in each panel.

ZOOM_CAL_START = pd.to_datetime("2015-09-01")
ZOOM_CAL_END   = pd.to_datetime("2015-09-30")

ZOOM_VAL_START = pd.to_datetime("2019-09-01")
ZOOM_VAL_END   = pd.to_datetime("2019-09-30")


# ── Panel drawing helper ───────────────────────────────────────────────────────

def draw_panel(ax, df, zoom_start, zoom_end, inset_pos, panel_label, show_xlabel=True):
    """Draw one calibration or validation panel onto *ax*."""

    # Metrics — with reservoir
    NSE_wr, RMSE_wr                    = nse(df["Obs"], df["With_Res"]),    rmse(df["Obs"], df["With_Res"])
    KGE_wr, r_wr,  beta_wr,  gamma_wr  = kge(df["Obs"], df["With_Res"])

    # Metrics — without reservoir
    NSE_nr, RMSE_nr                    = nse(df["Obs"], df["Without_Res"]), rmse(df["Obs"], df["Without_Res"])
    KGE_nr, r_nr,  beta_nr,  gamma_nr  = kge(df["Obs"], df["Without_Res"])

    # Main lines
    ax.plot(df["DateTime"], df["Obs"],
            label="Observed",                color="black",   linewidth=1.2, zorder=1, alpha=0.90)
    ax.plot(df["DateTime"], df["Without_Res"],
            label="Sim (without reservoir)", color="#2196F3",
            linestyle="--", linewidth=0.9,   zorder=2, alpha=0.85)
    ax.plot(df["DateTime"], df["With_Res"],
            label="Sim (with reservoir)",    color="#d62728",
            linestyle="-",  linewidth=1.0,   zorder=3, alpha=0.90)

    # x-axis ticks
    ax.set_xlim(df["DateTime"].min(), df["DateTime"].max())
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_minor_locator(mdates.MonthLocator())
    ax.tick_params(axis='x', which='minor', length=3, color='gray')
    ax.tick_params(axis='x', which='major', length=6)
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

    # y-axis
    data_max = max(df["Obs"].max(), df["With_Res"].max(), df["Without_Res"].max())
    ax.set_ylim(0, data_max * 1.05)
    y_lo, y_hi = ax.get_ylim()

    # Zoom box
    rect = plt.Rectangle(
        (mdates.date2num(zoom_start), y_lo),
        mdates.date2num(zoom_end) - mdates.date2num(zoom_start),
        y_hi - y_lo,
        linewidth=1.2, edgecolor='darkorange', facecolor='orange',
        alpha=0.08, zorder=4,
    )
    ax.add_patch(rect)
    for xd in [zoom_start, zoom_end]:
        ax.axvline(xd, color='darkorange', lw=0.9, ls='--', alpha=0.7, zorder=5)

    # Metric text boxes
    text_nr = (
        f"  Without Reservoir\n"
        f"NSE   = {NSE_nr:.3f}\n"
        f"RMSE  = {RMSE_nr:.3f}\n"
        f"KGE   = {KGE_nr:.3f}\n"
        f"r     = {r_nr:.3f}\n"
        f"β     = {beta_nr:.3f}\n"
        f"γ     = {gamma_nr:.3f}"
    )
    text_wr = (
        f"   With Reservoir  \n"
        f"NSE   = {NSE_wr:.3f}\n"
        f"RMSE  = {RMSE_wr:.3f}\n"
        f"KGE   = {KGE_wr:.3f}\n"
        f"r     = {r_wr:.3f}\n"
        f"β     = {beta_wr:.3f}\n"
        f"γ     = {gamma_wr:.3f}"
    )
    ax.text(0.78, 0.98, text_nr, transform=ax.transAxes, fontsize=6.5,
            va='top', ha='left',
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.9, edgecolor="#2196F3"))
    ax.text(0.78, 0.695, text_wr, transform=ax.transAxes, fontsize=6.5,
            va='top', ha='left',
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.9, edgecolor="#d62728"))

    # Panel label (outside axes, top-left — journal style)
    ax.set_title(panel_label, loc='left', fontsize=9.5, fontweight='bold', pad=4)

    # Axis labels
    if show_xlabel:
        ax.set_xlabel("Date / Time")
    ax.set_ylabel("Discharge (m³/s)")

    # Inset zoom panel
    axins = ax.inset_axes(inset_pos)
    axins.yaxis.set_label_position("right")
    axins.yaxis.tick_right()

    mask = (df["DateTime"] >= zoom_start) & (df["DateTime"] <= zoom_end)
    dz   = df[mask]

    axins.plot(dz["DateTime"], dz["Obs"],          color="black",   linewidth=1.0)
    axins.plot(dz["DateTime"], dz["Without_Res"],  color="#2196F3", linestyle="--", linewidth=0.8)
    axins.plot(dz["DateTime"], dz["With_Res"],     color="#d62728", linestyle="-",  linewidth=0.9)

    axins.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
    axins.xaxis.set_major_formatter(mdates.DateFormatter('%d\n%b'))
    axins.xaxis.set_minor_locator(mdates.DayLocator())
    axins.set_xlim(zoom_start, zoom_end)
    axins.set_ylabel("m³/s", fontsize=7)
    axins.grid(True, which='major', linestyle='', alpha=0.4)
    axins.patch.set_alpha(0.95)

    mark_inset(ax, axins, loc1=2, loc2=3, fc="none", ec="darkorange",
               lw=0.8, linestyle='--', alpha=0.7)

    return ax.get_legend_handles_labels()


# ── Main ───────────────────────────────────────────────────────────────────────

def main():
    plt.rcParams.update({
        'font.family':     'monospace',
        'font.size':       9,
        'axes.titlesize':  9.5,
        'axes.labelsize':  9,
        'xtick.labelsize': 8,
        'ytick.labelsize': 8,
        'legend.fontsize': 9,
    })

    # ── Load ──────────────────────────────────────────────────────────────────
    data_file = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Simulation_ZR\Calibration.xlsx"
    df = pd.read_excel(data_file)

    # ── Clean column names ────────────────────────────────────────────────────
    # Excel can silently add leading/trailing spaces, non-breaking spaces (0xA0),
    # or a BOM character (0xFEFF) to header cells — this strips all of them.
    df.columns = (
        df.columns
          .str.strip()
          .str.replace('\xa0',  '', regex=False)   # non-breaking space
          .str.replace('\ufeff', '', regex=False)  # BOM
    )

    # ── Diagnostic print (remove after confirming it works) ───────────────────
    print("Columns seen by pandas:", list(df.columns))

    # ── Guard: fail early with a helpful message if columns are still missing ─
    required = {'Zeit', 'Obs', 'With_Res', 'Without_Res'}
    missing  = required - set(df.columns)
    if missing:
        raise KeyError(
            f"\nRequired columns not found: {missing}"
            f"\nActual columns in the file: {list(df.columns)}"
            f"\nRename the Excel headers to match exactly, or update the "
            f"column names throughout this script."
        )

    # ── Parse datetime & drop rows with missing values ────────────────────────
    df['DateTime'] = pd.to_datetime(df['Zeit'], format="%d.%m.%Y %H:%M", errors='coerce')
    df = df.dropna(subset=['DateTime', 'Obs', 'With_Res', 'Without_Res'])

    # ── Split calibration / validation ────────────────────────────────────────
    CAL_START = pd.to_datetime("2011-11-01 00:00")
    CAL_END   = pd.to_datetime("2017-10-31 23:00")
    VAL_START = pd.to_datetime("2017-11-01 00:00")
    VAL_END   = pd.to_datetime("2022-10-31 23:00")

    df_cal = df[(df["DateTime"] >= CAL_START) & (df["DateTime"] <= CAL_END)].copy()
    df_val = df[(df["DateTime"] >= VAL_START) & (df["DateTime"] <= VAL_END)].copy()

    print(f"Calibration rows : {len(df_cal)}")
    print(f"Validation rows  : {len(df_val)}")

    # ── Figure: two panels stacked vertically ─────────────────────────────────
    fig, (ax_cal, ax_val) = plt.subplots(2, 1, figsize=(6.5, 8.5), sharex=False)
    fig.subplots_adjust(hspace=0.30)

    # Top panel — Calibration
    handles, labels = draw_panel(
        ax=ax_cal, df=df_cal,
        zoom_start=ZOOM_CAL_START, zoom_end=ZOOM_CAL_END,
        inset_pos=[0.27, 0.57, 0.38, 0.40],
        panel_label="a) Calibration (2011\u20132017)",
        show_xlabel=False,
    )

    # Bottom panel — Validation
    draw_panel(
        ax=ax_val, df=df_val,
        zoom_start=ZOOM_VAL_START, zoom_end=ZOOM_VAL_END,
        inset_pos=[0.27, 0.57, 0.38, 0.40],
        panel_label="b) Validation (2017\u20132022)",
        show_xlabel=True,
    )

    # Shared footer legend
    fig.legend(
        handles, labels,
        loc='lower center',
        bbox_to_anchor=(0.5, -0.01),
        ncol=3, fontsize=9.5,
        frameon=True, edgecolor='lightgray',
        fancybox=False, handlelength=2.0, columnspacing=2.0,
    )

    plt.savefig("Simulation_Cal_Val.png", dpi=300, bbox_inches='tight')
    plt.show()


if __name__ == "__main__":
    main()